## Basic ChatBot

In [2]:
from dotenv import load_dotenv
import os

from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, END, START
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_groq import ChatGroq

from langgraph.checkpoint.memory import MemorySaver

from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper

load_dotenv()

C:\Users\Acer\AppData\Local\Temp\ipykernel_11412\2892083220.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchRun


True

In [3]:
wiki_wrapper = WikipediaAPIWrapper(top_k_results=2 ,doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=wiki_wrapper, description="Find Information from Wikipedia Pages")

In [4]:
duck_duck_go = DuckDuckGoSearchRun(
    name="web_search_on_Duck_Duck_Go",
    description="Search the web for current events, recent news, live scores, and real-time information."
    )

In [6]:
tools = [wiki, duck_duck_go]

In [19]:
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

In [30]:
short_term_memory = MemorySaver()

In [20]:
model = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.2)

lllm_with_tools = model.bind_tools(tools)

In [21]:
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [22]:
def tool_calling_llm(state: State) -> State:
    return {'messages': [lllm_with_tools.invoke(state['messages'])  ]}

In [31]:
graph = StateGraph(State)

graph.add_node('tool_calling_llm', tool_calling_llm)
graph.add_node('tools', ToolNode(tools))

graph.set_entry_point('tool_calling_llm')
graph.add_conditional_edges('tool_calling_llm', tools_condition)
graph.add_edge('tools', 'tool_calling_llm')

build = graph.compile(checkpointer=short_term_memory)

In [33]:
config = {"configurable": {"thread_id": 1}}
messages = build.invoke({'messages': HumanMessage("Hii, my name is Vishal")}, config=config)

for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Hii, my name is Vishal
================================ Human Message =================================

Hii, my name is Vishal
================================== Ai Message ==================================

Hi Vishal! Nice to meet you. How can I help you today?


In [34]:
messages = build.invoke({'messages': HumanMessage("Do you remember me")}, config=config)

for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Hii, my name is Vishal
================================ Human Message =================================

Hii, my name is Vishal
================================== Ai Message ==================================

Hi Vishal! Nice to meet you. How can I help you today?
================================ Human Message =================================

Do you remember me
================================== Ai Message ==================================

Yes, I remember! Your name is **Vishal**.

How can I help you today?


## Streaming

In [47]:
config = {"configurable": {"thread_id": 2}}
for chunk in build.stream({'messages': HumanMessage("What is Football ?")}, config=config):
    print(chunk)


{'tool_calling_llm': {'messages': [AIMessage(content='The term **Football** refers to a family of sports played with a ball, where the objective is to score goals. However, the specific rules, equipment, and popularity vary significantly depending on the region of the world.\n\nHere are the three main types of football:\n\n### 1. Association Football (Soccer)\n*   **Global Dominance:** This is the most popular sport in the world, played in over 200 countries.\n*   **Also Known As:** "Football" in most of the world (Europe, South America, Africa, Asia, Oceania). In the United States and Canada, it is commonly called **Soccer** to distinguish it from American football.\n*   **Key Features:**\n    *   **Ball:** Round.\n    *   **Teams:** Two teams of 11 players.\n    *   **Objective:** Score by kicking the ball into the opponent’s net.\n    *   **Hands:** Players cannot use their hands or arms (except goalkeepers within their penalty area).\n    *   **Governing Body:** FIFA (Fédération In

In [ ]:
config = {"configurable": {"thread_id": 13}}
async for chunk in build.astream_events({'messages': HumanMessage("Who is prime minister of india")}, config=config, version='v2'):
    print(chunk)